# Generate protein prizes from TPS input files - November 4, 2016

## Initialize

In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
import os

in_dir = os.path.join('..','tps','data','timeseries')
p_first_file = os.path.join(in_dir, 'p-values-first.tsv')
p_prev_file = os.path.join(in_dir, 'p-values-prev.tsv')
map_file = os.path.join(in_dir, 'peptide-mapping.tsv')
out_file = os.path.join(in_dir, 'generated-prizes.tsv')

# Only used to verify the calculated protein prizes
prize_file = os.path.join('..','tps','data','pcsf','egfr-prizes.txt')

## Build the peptide-protein map
No header row.  Supports mapping a peptide to a set of pipe-delimited proteins.  Also allow a peptide to appear multiple times and include all proteins mapped to that peptide in any line.

In [2]:
pep_prot_map = defaultdict(set)
unique_prots = set()

with open(map_file) as map_f:
    for line in map_f:
        parts = line.strip().split('\t')
        assert len(parts) == 2, 'Expected tab-delimited peptide and protein (or pipe-delimited proteins) on each line'
        
        pep = parts[0]
        prots = parts[1].split('|')
        
        # Add all proteins that map to the peptide in this line to any existing proteins that mapped to the peptide
        # in previous lines
        pep_prot_map[pep].update(prots)
        # Add the proteins to the set of all observed proteins
        unique_prots.update(prots)

print 'Loaded {} peptides that map to {} unique proteins'.format(len(pep_prot_map), len(unique_prots))

Loaded 2918 peptides that map to 1556 unique proteins


## Load the two significance scores files
From https://github.com/koksal/tps:
> Significance scores can be naively computed with t-tests
comparing the phosphorylation intensity at each time point and the first time point.
A preferable option is to account for the comparisons of multiple pairs of time
points using [Tukey's Honest Significant Difference test], which is implemented as [TukeyHSD]
in R. This test compares all pairs of time points, from which the comparisons to the
first time point can be extracted. This file should not contain a header row, and if a
header row is provided it should be commented out with a leading **#** character. If
there are *t* time points in the `--timeseries <file>`, this file should contain
*t* - 1 significance score columns.  Missing values and **N/A** are not allowed and
should be replaced by placeholder scores of **1.0**.

In [3]:
first_df = pd.read_csv(p_first_file, sep='\t', comment='#', header=None, index_col=0)
prev_df = pd.read_csv(p_prev_file, sep='\t', comment='#', header=None, index_col=0)
first_shape = first_df.shape
assert first_shape == prev_df.shape, 'First and previous score files must have the same number of peptides and time points'

assert not first_df.isnull().values.any(), 'First scores file contains N/A values.  Replace with 1.0'
assert not prev_df.isnull().values.any(), 'Previous scores file contains N/A values.  Replace with 1.0'

print 'Loaded {} peptides and {} scores in each file'.format(first_shape[0], first_shape[1])

first_df.head()

Loaded 1068 peptides and 7 scores in each file


,1,2,3,4,5,6,7
0,,,,,,,
-.n[305.21]MRK[432.30]ET[181.01]PPPLVPPAAR.E,0.999811,0.484909,1.000000,0.374909,0.999975,0.976976,0.874649
-.n[305.21]MSTTAGY[243.03]LARR.A,0.207988,0.212137,0.296223,0.374641,0.695874,0.983827,0.423177
K.n[305.21]AAASGQPRPEMQC[160.03]PAEHEEDMY[243.03]RAADEIEK[432.30].E,0.005700,0.400535,0.389710,0.746805,0.990944,0.999436,1.000000
K.n[305.21]AAGPAS[167.00]PPEEDPER.T,0.999995,0.999996,0.998033,0.987489,0.963331,0.999972,0.997236
K.n[305.21]AALGLQDS[167.00]DDEDAAVDIDEQIESMFNSK[432.30].K,0.738649,0.016827,0.189344,0.009719,0.218471,0.499859,0.932410


## Merge the two significance scores files

In [4]:
merged_df = pd.concat([first_df, prev_df], axis=1, join='outer')
merged_shape = merged_df.shape
assert merged_shape[0] == first_shape[0], 'First and previous significance scores contain different peptides'
assert merged_shape[1] == 2*first_shape[1], 'Unexpected number of significance scores after merging first and previous significance scores'

## Compute a prize for each peptide
The prize is -log10(min p-value across all time points)

In [5]:
def calc_prize(row):
    return -np.log10(min(row))

merged_df['prize'] = merged_df.apply(calc_prize, axis=1)
merged_df.head()

,1,2,3,4,5,6,7,1,2,3,4,5,6,7,prize
0,,,,,,,,,,,,,,,
-.n[305.21]MRK[432.30]ET[181.01]PPPLVPPAAR.E,0.999811,0.484909,1.000000,0.374909,0.999975,0.976976,0.874649,0.999811,0.729535,0.571277,0.452852,0.241322,0.903082,0.376328,0.617402
-.n[305.21]MSTTAGY[243.03]LARR.A,0.207988,0.212137,0.296223,0.374641,0.695874,0.983827,0.423177,0.207988,1.000000,0.999997,1.000000,0.998663,0.989551,0.115603,0.937030
K.n[305.21]AAASGQPRPEMQC[160.03]PAEHEEDMY[243.03]RAADEIEK[432.30].E,0.005700,0.400535,0.389710,0.746805,0.990944,0.999436,1.000000,0.005700,0.259488,1.000000,0.997489,0.990042,0.999987,0.999629,2.244160
K.n[305.21]AAGPAS[167.00]PPEEDPER.T,0.999995,0.999996,0.998033,0.987489,0.963331,0.999972,0.997236,0.999995,1.000000,0.986135,0.820275,0.999999,0.995613,0.999968,0.086041
K.n[305.21]AALGLQDS[167.00]DDEDAAVDIDEQIESMFNSK[432.30].K,0.738649,0.016827,0.189344,0.009719,0.218471,0.499859,0.932410,0.738649,0.259542,0.843098,0.685057,0.631296,0.997939,0.091643,2.012386


## Compute a prize for each protein
Take the maximum prize across all peptides that map to the protein.  The same peptide can contribute to the prize of multiple proteins if it maps to multiple protiens.  Write the prizes to a file.

In [6]:
# Default prize is 0.0
prot_prizes = defaultdict(float)

for index, row in merged_df.iterrows():
    # index is the peptide
    assert index in pep_prot_map, 'No protein mapping for {}'.format(index)
    prots = pep_prot_map[index]
    # The peptide can map to a set of proteins
    for prot in prots:
        # Keep the larger of the previous best prize for this protein and the current prize
        prot_prizes[prot] = max(prot_prizes[prot], row['prize'])

print 'Writing prizes for {} unique proteins to {}'.format(len(prot_prizes), out_file)

with open(out_file, 'w') as o_f:
    for prot in sorted(prot_prizes.keys()):
        o_f.write('{}\t{}\n'.format(prot, prot_prizes[prot]))

Writing prizes for 701 unique proteins to ..\tps\data\timeseries\generated-prizes.tsv


## Compare the new prizes with the existing prizes

In [7]:
existing_prize_map = dict()
with open(prize_file) as map_f:
    for line in map_f:
        parts = line.strip().split('\t')
        assert len(parts) == 2, 'Expected tab-delimited protein and prize on each line'
        
        prot = parts[0]
        prize = float(parts[1])
        existing_prize_map[prot] = prize

assert set(prot_prizes.keys()) == set(existing_prize_map.keys()), 'Different sets of proteins with prizes'
for prot in prot_prizes.keys():
    assert np.isclose(prot_prizes[prot], existing_prize_map[prot], atol=1e-5), 'Prizes do not match for {}'.format(prot)

print 'Prizes in {} match {}'.format(out_file, prize_file)

if os.path.exists(out_file):
    print 'Deleting {}'.format(out_file)
    os.remove(out_file)

Prizes in ..\tps\data\timeseries\generated-prizes.tsv match ..\tps\data\pcsf\egfr-prizes.txt
Deleting ..\tps\data\timeseries\generated-prizes.tsv
